In [6]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import pandas as pd
# from upsetplot import from_memberships, UpSet
import matplotlib.pyplot as plt

import math
import json

In [7]:
df = pd.read_csv("../../data/papers.csv")
total_papers = len(df)
df.columns

Index(['Title', 'Database', 'Year', 'Month', 'Journal', 'Paper Type',
       'Data Source', 'Data Country', 'Data Domain', 'Data Language',
       'Data Availability', 'Link to Data', 'Data Details', 'Size',
       'Number of Prompts', 'Medical Application', 'Task Type', 'Topic',
       'Note', 'Bias Evaluation Metric', 'Bias Definition', 'Conclusions',
       'Race / Ethnic Bias', 'Language Bias', 'Age Bias', 'Gender Bias',
       'Other Bias', 'LGBTQ+ Bias', 'Disability Bias',
       'Geography / Cultural Bias', 'Evaluated LLMs',
       'Reference Standard \r\n(Human / Model / System / Physician)',
       'Patient Inclusion\r\n(Yes / No)', 'Has Debiasing',
       'Debias - Focus\r\n(Data/Train/Inference)', 'Debias-details'],
      dtype='str')

In [8]:
paper_conclusions_lst = df["Conclusions"].tolist()
conclusions_mps = {}

for paper in paper_conclusions_lst:
    if isinstance(paper, float):
        continue

    # comma-separated list like: "Female: Yes (...), Male: No (...)"
    for conclusion in str(paper).split(","):
        try:
            # drop parenthetical notes
            value = re.sub(r"\s*\([^)]*\)", "", conclusion).strip()

            key, val = (part.strip() for part in value.split(":", 1))
            if not key or not val:
                continue

            key_norm = key.strip()
            yn = val.strip().lower().split(None, 1)[0]  # first token

            if key_norm not in conclusions_mps:
                conclusions_mps[key_norm] = {"yes": 0, "no": 0}

            if yn == "yes":
                conclusions_mps[key_norm]["yes"] += 1
            elif yn == "no":
                conclusions_mps[key_norm]["no"] += 1

        except ValueError:
            continue

conclusions_mps

{'Female': {'yes': 20, 'no': 17},
 'Male': {'yes': 11, 'no': 25},
 'White': {'yes': 6, 'no': 18},
 'Non White': {'yes': 0, 'no': 1},
 'African American': {'yes': 2, 'no': 2},
 'Caucasion': {'yes': 1, 'no': 0},
 'Ashkenazi Jews': {'yes': 1, 'no': 0},
 'Mediterranean/Middle Eastern': {'yes': 1, 'no': 0},
 'East Asians': {'yes': 1, 'no': 0},
 'Asians': {'yes': 0, 'no': 1},
 'Black': {'yes': 15, 'no': 8},
 'Whites': {'yes': 0, 'no': 1},
 'Hispanic': {'yes': 8, 'no': 7},
 '75yo': {'yes': 1, 'no': 0},
 '25yo': {'yes': 0, 'no': 1},
 'Asian': {'yes': 10, 'no': 6},
 'Arab': {'yes': 2, 'no': 0},
 'Multiracial': {'yes': 1, 'no': 0},
 'Native American or Indigenous': {'yes': 1, 'no': 0},
 'Middle Eastern': {'yes': 2, 'no': 0},
 'Unhoused': {'yes': 2, 'no': 0},
 'retired': {'yes': 0, 'no': 1},
 'student': {'yes': 0, 'no': 1},
 'unemployed': {'yes': 0, 'no': 1},
 'bisexual': {'yes': 1, 'no': 0},
 'gay/lesbian': {'yes': 1, 'no': 0},
 'heterosexual': {'yes': 1, 'no': 0},
 'transgender': {'yes': 1, 'no

In [9]:
groups = {
    'Female': ['Female', 'female', 'Transgender Female', 
                'Women', 'LMIC women and marginalized groups',
                'Woman', 'Transgender woman'],
    'Male': ['Male', 'male', 'Transgender Male', 'Men', 
            'Transgender man', 'Man'],
    'Other': ['Gender Nonconforming', ]
}

gender_conclusions = {}

# Combine each group
for new_name, categories in groups.items():
    yes_total = sum(conclusions_mps[cat]['yes'] for cat in categories)
    no_total = sum(conclusions_mps[cat]['no'] for cat in categories)
    
    gender_conclusions[new_name] = {'yes': yes_total, 'no': no_total}
    
    # Delete original categories
    for cat in categories:
        del conclusions_mps[cat]


In [ ]:
with open("gender_conclusions/gender_conclusions.json", "w") as file:
    json.dump(gender_conclusions, file, indent=4, sort_keys=True)
